# 02. Análisis exploratorio y estadística descriptiva


### Objetivo
Documentar propiedades a partir de tablas consolidadas.



## Preparación del entorno y lectura reproducible
El notebook busca el directorio `data/processed` generado en la etapa anterior, en su ausencia, permite ejecutarse con los dataset colocados junto al notebook o dentro de una carpeta indicada. No requiere los archivos originales.

Muestra cantidad de registros y cantidad de variables en los dataset trabajados.

In [31]:

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

FILES = [
    "column_quality.csv", "genre_frequency.csv", "key_checks.csv",
    "movie_genres.csv", "movies_consolidated.csv", "movies_without_ratings.csv",
    "movies_without_tags.csv", "post_validation.csv", "quality_overview.csv",
    "rating_aggregates_by_movie.csv", "rating_distribution.csv",
    "referential_integrity.csv", "tag_aggregates_by_movie.csv",
    "title_duplicates_exact.csv"
]


DATA_DIR_OVERRIDE = None
cwd = Path.cwd().resolve()
candidates = ([Path(DATA_DIR_OVERRIDE).expanduser().resolve()] if DATA_DIR_OVERRIDE else []) + [
    p / "data" / "processed" for p in (cwd, *cwd.parents)
] + [cwd, cwd / "data" / "processed", cwd / "archivos", Path("/mnt/data")]
DATA_DIR = next((p for p in candidates if p.is_dir() and all((p / f).is_file() for f in FILES)), None)
if DATA_DIR is None:
    raise FileNotFoundError("No se localizaron los dataset. Configura DATA_DIR_OVERRIDE con la carpeta donde se encuentran.")

tables = {Path(f).stem: pd.read_csv(DATA_DIR / f, low_memory=False) for f in FILES}
movies = tables["movies_consolidated"]
movie_genres = tables["movie_genres"]
rating_dist = tables["rating_distribution"].copy()
print("Directorio de datos:", DATA_DIR)
#print("Bases cargadas:", len(tables))
display(pd.DataFrame([{"archivo": f"{k}.csv", "registros": len(v), "variables": v.shape[1]} for k,v in tables.items()]))
assert len(tables) == 14
assert movies["movieId"].is_unique, "movies_consolidated debe tener una película por fila"
assert not movie_genres.duplicated(["movieId", "genre"]).any(), "Duplicación película-género"

Directorio de datos: C:\Users\ari1\Documents\Diplomado\Modulo_1\Proyecto_Integrador_M1\data\processed


,archivo,registros,variables
0,column_quality.csv,14,7
1,genre_frequency.csv,20,3
2,key_checks.csv,4,5
3,movie_genres.csv,154170,2
4,movies_consolidated.csv,87585,20
5,movies_without_ratings.csv,3153,3
6,movies_without_tags.csv,36262,3
7,post_validation.csv,6,2
8,quality_overview.csv,4,8
9,rating_aggregates_by_movie.csv,84432,7


## 1. Entendimiento de los datos y registros relevantes
Las películas son entidades del catálogo, las puntuaciones son evaluaciones explícitas de usuarios y las etiquetas describen aportaciones textuales. Los identificadores de MovieLens, IMDb y TMDb son **identificadores **, no magnitudes, se consideran cualitativa nominal. `rating_count` mide cuántas evaluaciones recibió una película; `rating_mean` sólo existe si recibió al menos una. La ausencia de evaluaciones o etiquetas no equivale a una puntuación de cero ni a una opinión negativa.


In [19]:
# 2. COBERTURA, INTEGRIDAD Y REPRESENTATIVIDAD
q = tables["quality_overview"]
display(q)
for col in ["has_ratings", "has_tags", "has_imdb_id", "has_tmdb_id"]:
    if col in movies.columns:
        print(f"{col}: {movies[col].astype('boolean').mean()*100:.2f}% de las películas")
print("Calificaciones totales (tabla de frecuencias):", int(rating_dist["frequency"].sum()))
print("Películas distintas:", movies["movieId"].nunique())
print("Películas sin calificaciones:", int((movies["rating_count"] == 0).sum()))
print("Películas sin etiquetas:", int((movies["tag_application_count"] == 0).sum()))
print("Géneros distintos:", movie_genres["genre"].nunique())
print("Media global de evaluaciones por evento:",
      np.average(rating_dist["rating"], weights=rating_dist["frequency"]))
print("Media no ponderada de medias por película:", movies["rating_mean"].mean())
print("Las dos medias anteriores responden preguntas distintas: evento vs. película.")

ranking = movies.loc[movies["rating_count"] >= 100, ["movieId", "title", "rating_count", "rating_mean", "tag_application_count"]]
print("Películas con mayor número de evaluaciones (sin inferir calidad):")
display(ranking.sort_values("rating_count", ascending=False).head(10))
print("Películas con media más elevada, restringiendo a >=100 evaluaciones:")
display(ranking.sort_values(["rating_mean", "rating_count"], ascending=False).head(10))

,dataset,records,columns,distinct_values_sum,full_row_duplicates,null_cells,completeness_pct,memory_mb
0,movies,87585,3,176765,0,0,100.0000,11.9166
1,ratings,32000204,4,26261913,0,0,100.0000,610.3556
2,tags,2000072,4,1499400,0,17,99.9998,145.2141
3,links,87585,3,262595,0,124,99.9528,1.8377


has_ratings: 96.40% de las películas
has_tags: 58.60% de las películas
has_imdb_id: 100.00% de las películas
has_tmdb_id: 99.86% de las películas
Calificaciones totales (tabla de frecuencias): 32000204
Películas distintas: 87585
Películas sin calificaciones: 3153
Películas sin etiquetas: 36262
Géneros distintos: 20
Media global de evaluaciones por evento: 3.5403956487277393
Media no ponderada de medias por película: 3.005082385038019
Las dos medias anteriores responden preguntas distintas: evento vs. película.
Películas con mayor número de evaluaciones (sin inferir calidad):


,movieId,title,rating_count,rating_mean,tag_application_count
314,318,"Shawshank Redemption, The (1994)",102929,4.4046,4892
351,356,Forrest Gump (1994),100296,4.0527,3581
292,296,Pulp Fiction (1994),98409,4.1970,6697
2480,2571,"Matrix, The (1999)",93808,4.1564,5722
585,593,"Silence of the Lambs, The (1991)",90330,4.1484,3469
257,260,Star Wars: Episode IV - A New Hope (1977),85010,4.0998,6519
2867,2959,Fight Club (1999),77332,4.2288,5532
475,480,Jurassic Park (1993),75233,3.6986,1510
522,527,Schindler's List (1993),73849,4.2370,2059
4888,4993,"Lord of the Rings: The Fellowship of the Ring,...",73122,4.0921,2550


Películas con media más elevada, restringiendo a >=100 evaluaciones:


,movieId,title,rating_count,rating_mean,tag_application_count
46252,171011,Planet Earth II (2016),1956,4.4468,96
40985,159817,Planet Earth (2006),2948,4.4444,103
46103,170705,Band of Brothers (2001),2811,4.4265,234
314,318,"Shawshank Redemption, The (1994)",102929,4.4046,4892
46487,171495,Cosmos,615,4.3301,167
840,858,"Godfather, The (1972)",66440,4.3170,2310
61090,202439,Parasite (2019),11670,4.3123,3052
50126,179135,Blue Planet II (2017),1163,4.3001,74
59131,198185,Twin Peaks (1989),1140,4.2987,146
68471,220528,Twelve Angry Men (1954),449,4.2862,8


### Sesgos y límites de inferencia
- **Autoselección:** quienes califican películas no constituyen una muestra aleatoria de todas las personas ni de todos los espectadores.
- **Popularidad y supervivencia de actividad:** las películas con mayor exposición acumulan más observaciones; medias de títulos con pocos votos son menos estables.
- **Sesgo de cobertura:** los títulos sin evaluaciones participan en el catálogo, pero no en la distribución de notas observadas.
- **Granularidad y ponderación:** una media de medias por película pondera todos los títulos igual; la media global de evaluaciones pondera por número de eventos.
- **Géneros multilabel:** una película puede pertenecer a varios géneros; sumar las frecuencias de géneros no produce el total de películas.
- **No causalidad:** una correlación entre actividad y puntuación no demuestra que una variable genere la otra.

## 2. Clasificación de variables, escalas de medición y comparaciones permitidas

Se diferencian **tipo estadístico** y **escala**. Para una nominal son válidas igualdad, diferencias de proporciones y moda, no promedios. Una ordinal añade orden, mediana y cuantiles cuando se trata de puntuaciones ordenadas. La escala de intervalo permite diferencias numéricas pero no interpretaciones de tipo “el doble”; razón añade un cero absoluto y cocientes interpretables. Las puntuaciones `rating` son **ordinales discretas** en sentido estricto; analizar sus medias como *intervalo aproximado* constituye un supuesto práctico explícito. Una marca booleana es nominal dicotómica aunque se almacene como `bool`. Los identificadores numéricos siguen siendo nominales.

La desviación estándar, CV, asimetría, curtosis y Pearson no se aplican a IDs, textos ni indicadores categóricos. En las variables de conteo, un cero sí indica ausencia de actividad; en las medias de puntuación, cero no es un valor admisible en la escala.

In [20]:
# 3. DICCIONARIO ESTADÍSTICO DOCUMENTADO
metadata = {
"movieId": ("Cualitativa nominal", "Nominal", "ID de película", "Igualdad, unión, conteos de películas"),
"title": ("Cualitativa nominal", "Nominal", "Título", "Frecuencias y títulos duplicados"),
"genres": ("Cualitativa nominal multilabel", "Nominal", "Lista de géneros", "Pertenencia y proporciones por categoría"),
"genres_original": ("Cualitativa nominal multilabel", "Nominal", "Géneros originales", "Comparación de categorías"),
"genre": ("Cualitativa nominal", "Nominal", "Género individual", "Frecuencias y cobertura por género"),
"imdbId": ("Cualitativa nominal", "Nominal", "ID IMDb", "Unicidad y enlaces"),
"tmdbId": ("Cualitativa nominal", "Nominal", "ID TMDb", "Unicidad y enlaces"),
"rating": ("Cualitativa ordinal (valores discretos)", "Ordinal; intervalo aproximado para medias", "Puntos (0.5–5)", "Orden, frecuencias, cuartiles; media bajo supuesto"),
"rating_count": ("Cuantitativa discreta", "Razón", "Número de evaluaciones", "Diferencias, cocientes, cuantiles, dispersión"),
"rating_users": ("Cuantitativa discreta", "Razón", "Usuarios que evaluaron", "Diferencias, cocientes, dispersión"),
"rating_mean": ("Cuantitativa continua derivada", "Intervalo aproximado", "Puntos por película", "Diferencias; media/correlación con cautela"),
"rating_std": ("Cuantitativa continua derivada", "Razón (magnitud de dispersión)", "Puntos por película", "Comparación de dispersión; no aplica con n<2"),
"rating_min": ("Cuantitativa derivada discreta", "Ordinal", "Puntos", "Orden y comparación de extremos"),
"rating_max": ("Cuantitativa derivada discreta", "Ordinal", "Puntos", "Orden y comparación de extremos"),
"tag_application_count": ("Cuantitativa discreta", "Razón", "Etiquetas por película", "Frecuencias, cocientes y dispersión"),
"non_null_tag_count": ("Cuantitativa discreta", "Razón", "Etiquetas con texto", "Frecuencias, cocientes y dispersión"),
"distinct_tags": ("Cuantitativa discreta", "Razón", "Etiquetas únicas por película", "Conteos y comparación de diversidad textual"),
"tag_users": ("Cuantitativa discreta", "Razón", "Usuarios con etiquetas", "Conteos y comparaciones"),
"has_ratings": ("Cualitativa nominal dicotómica", "Nominal", "Sí/no", "Proporciones y tablas cruzadas"),
"has_tags": ("Cualitativa nominal dicotómica", "Nominal", "Sí/no", "Proporciones y tablas cruzadas"),
"has_imdb_id": ("Cualitativa nominal dicotómica", "Nominal", "Sí/no", "Cobertura de identificadores"),
"has_tmdb_id": ("Cualitativa nominal dicotómica", "Nominal", "Sí/no", "Cobertura de identificadores"),
}
dictionary = pd.DataFrame([{
    "variable": c, "tipo_estadístico": metadata.get(c, ("Revisar", "Revisar", "", ""))[0],
    "escala": metadata.get(c, ("Revisar", "Revisar", "", ""))[1],
    "unidad/significado": metadata.get(c, ("", "", "Revisar", ""))[2],
    "comparaciones_permitidas": metadata.get(c, ("", "", "", "Revisar"))[3],
    "tipo_python": str(movies[c].dtype)
} for c in movies.columns])
display(dictionary)
assert not dictionary["tipo_estadístico"].eq("Revisar").any(), "Clasifica cualquier columna nueva antes de analizarla"

,variable,tipo_estadístico,escala,unidad/significado,comparaciones_permitidas,tipo_python
0,movieId,Cualitativa nominal,Nominal,ID de película,"Igualdad, unión, conteos de películas",int64
1,title,Cualitativa nominal,Nominal,Título,Frecuencias y títulos duplicados,object
2,genres,Cualitativa nominal multilabel,Nominal,Lista de géneros,Pertenencia y proporciones por categoría,object
3,genres_original,Cualitativa nominal multilabel,Nominal,Géneros originales,Comparación de categorías,object
4,imdbId,Cualitativa nominal,Nominal,ID IMDb,Unicidad y enlaces,int64
5,tmdbId,Cualitativa nominal,Nominal,ID TMDb,Unicidad y enlaces,float64
6,rating_count,Cuantitativa discreta,Razón,Número de evaluaciones,"Diferencias, cocientes, cuantiles, dispersión",int64
7,rating_mean,Cuantitativa continua derivada,Intervalo aproximado,Puntos por película,Diferencias; media/correlación con cautela,float64
8,rating_std,Cuantitativa continua derivada,Razón (magnitud de dispersión),Puntos por película,Comparación de dispersión; no aplica con n<2,float64
9,rating_min,Cuantitativa derivada discreta,Ordinal,Puntos,Orden y comparación de extremos,float64


## 3. Distribuciones de frecuencia
Para variables nominales se presentan frecuencias absolutas y relativas. Para conteos cuantitativos se incluyen intervalos interpretables definidos de antemano.

In [21]:
# 4. FRECUENCIAS CATEGÓRICAS Y ORDINALES
print("Distribución completa de calificaciones por evento:")
rd = rating_dist.sort_values("rating").copy()
rd["porcentaje_recalculado"] = rd["frequency"] / rd["frequency"].sum() * 100
display(rd)
print("Frecuencia de géneros: una película puede aparecer en varios géneros")
g = movie_genres.groupby("genre", dropna=False)["movieId"].nunique().sort_values(ascending=False).rename("peliculas").reset_index()
g["porcentaje_catalogo"] = 100 * g["peliculas"] / movies["movieId"].nunique()
display(g)
for flag in ["has_ratings", "has_tags", "has_imdb_id", "has_tmdb_id"]:
    d = movies[flag].value_counts(dropna=False).rename_axis(flag).reset_index(name="frecuencia")
    d["porcentaje"] = 100 * d["frecuencia"] / len(movies)
    print(flag)
    display(d)

bins = [-0.1, 0, 1, 5, 20, 100, 1000, np.inf]
labels = ["0", "1", "2–5", "6–20", "21–100", "101–1000", ">1000"]
for field in ["rating_count", "tag_application_count"]:
    d = pd.cut(movies[field], bins=bins, labels=labels).value_counts(sort=False)
    display(pd.DataFrame({"variable": field, "intervalo": d.index.astype(str), "frecuencia": d.values,
                          "porcentaje": d.values / len(movies) * 100}))
print("La suma de porcentajes por género puede superar 100% por clasificación múltiple.")

Distribución completa de calificaciones por evento:


,rating,frequency,percentage,porcentaje_recalculado
0,0.5000,525132,1.6410,1.6410
1,1.0000,946675,2.9583,2.9583
2,1.5000,531063,1.6596,1.6596
3,2.0000,2028622,6.3394,6.3394
4,2.5000,1685386,5.2668,5.2668
5,3.0000,6054990,18.9217,18.9217
6,3.5000,4290105,13.4065,13.4065
7,4.0000,8367654,26.1488,26.1488
8,4.5000,2974000,9.2937,9.2937
9,5.0000,4596577,14.3642,14.3642


Frecuencia de géneros: una película puede aparecer en varios géneros


,genre,peliculas,porcentaje_catalogo
0,Drama,34175,39.0192
1,Comedy,23124,26.4018
2,Thriller,11823,13.4989
3,Romance,10369,11.8388
4,Action,9668,11.0384
5,Documentary,9363,10.6902
6,Horror,8654,9.8807
7,(no genres listed),7080,8.0836
8,Crime,6976,7.9648
9,Adventure,5402,6.1677


has_ratings


,has_ratings,frecuencia,porcentaje
0,True,84432,96.4001
1,False,3153,3.5999


... (5 outputs left out)

,has_tmdb_id,frecuencia,porcentaje
0,True,87461,99.8584
1,False,124,0.1416


,variable,intervalo,frecuencia,porcentaje
0,rating_count,0,3153,3.5999
1,rating_count,1,18607,21.2445
2,rating_count,2–5,25420,29.0232
3,rating_count,6–20,17547,20.0343
4,rating_count,21–100,10713,12.2315
5,rating_count,101–1000,7750,8.8485
6,rating_count,>1000,4395,5.0180


,variable,intervalo,frecuencia,porcentaje
0,tag_application_count,0,36262,41.4021
1,tag_application_count,1,7682,8.7709
2,tag_application_count,2–5,18056,20.6154
3,tag_application_count,6–20,14519,16.5770
4,tag_application_count,21–100,7193,8.2126
5,tag_application_count,101–1000,3630,4.1445
6,tag_application_count,>1000,243,0.2774


La suma de porcentajes por género puede superar 100% por clasificación múltiple.


## 4. Medidas centrales, de variabilidad y robustez

Para conteos (escala de razón) se calculan media, mediana, moda, varianza **muestral** (`ddof=1`), desviación estándar muestral, cuartiles, rango intercuartílico `Q3−Q1`, y coeficiente de variación `DE/media` cuando la media es positiva. El **CV** carece de sentido para identificadores, medias de calificaciones en escala sin cero absoluto y variables cuyo promedio sea cero. 

El IQR concentra la amplitud del 50% central de observaciones; es relativamente resistente a extremos. Como indicador de sensibilidad se compara la media original con una media recortada simétricamente al 5% y con la mediana. La media recortada **no sustituye ni elimina** registros originales.

In [22]:
# 5. DESCRIPTIVOS ROBUSTOS PARA CONTEOS A NIVEL PELÍCULA
def trimmed_mean(x, proportion=0.05):
    v = np.sort(pd.to_numeric(pd.Series(x), errors="coerce").dropna().to_numpy(dtype=float))
    k = int(len(v) * proportion)
    return float(v[k: len(v)-k].mean()) if len(v) > 2*k else np.nan

def descriptive(x, name, cv=True):
    s = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
    if s.empty: return {"variable": name, "n": 0}
    q1, q3 = s.quantile([0.25, 0.75]); mu = s.mean(); sd = s.std(ddof=1)
    modes = s.mode()
    return {"variable": name, "n": int(len(s)), "media": mu, "mediana": s.median(),
            "moda": modes.iloc[0] if len(modes) else np.nan,
            "modas_totales": len(modes), "varianza_muestral": s.var(ddof=1),
            "desviacion_muestral": sd, "q1": q1, "q3": q3, "IQR": q3-q1,
            "mínimo": s.min(), "máximo": s.max(),
            "CV_pct": 100*sd/mu if cv and mu>0 else np.nan,
            "media_recortada_5pct": trimmed_mean(s),
            "cambio_media_recorte_pct": 100*(trimmed_mean(s)-mu)/mu if mu != 0 else np.nan}

count_vars = ["rating_count", "rating_users", "tag_application_count", "non_null_tag_count", "distinct_tags", "tag_users"]
count_stats = pd.DataFrame([descriptive(movies[c],c) for c in count_vars])
display(count_stats.set_index("variable").T)
print("CV: cuanto mayor es, más dispersión relativa en una misma medida de razón; no existe umbral universal.")
print("La comparación media/mediana y el cambio por recorte diagnostican sensibilidad, no prueban una distribución.")

variable,rating_count,rating_users,tag_application_count,non_null_tag_count,distinct_tags,tag_users
n,"87,585.0000","87,585.0000","87,585.0000","87,585.0000","87,585.0000","87,585.0000"
media,365.3617,365.3617,22.8358,22.8356,12.4088,3.9966
mediana,5.0000,5.0000,1.0000,1.0000,1.0000,1.0000
moda,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000
modas_totales,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
varianza_muestral,"6,483,784.1556","6,483,784.1556","15,545.3252","15,545.2835","1,991.1156",400.9613
desviacion_muestral,"2,546.3276","2,546.3276",124.6809,124.6807,44.6219,20.0240
q1,2.0000,2.0000,0.0000,0.0000,0.0000,0.0000
q3,23.0000,23.0000,7.0000,7.0000,7.0000,2.0000
IQR,21.0000,21.0000,7.0000,7.0000,7.0000,2.0000


CV: cuanto mayor es, más dispersión relativa en una misma medida de razón; no existe umbral universal.
La comparación media/mediana y el cambio por recorte diagnostican sensibilidad, no prueban una distribución.


### Descriptivos de las calificaciones y distinción entre unidades
La escala de calificación es discreta y ordenada. La media y la varianza se muestran bajo la convención habitual de tratar los saltos de 0.5 como intervalos comparables. Los descriptivos **por evento** usan frecuencias ponderadas; los descriptivos **por película** usan una observación por título con evaluaciones. No debe interpretarse el CV de estas puntuaciones como razón relativa.

In [23]:
# 6. ESTADÍSTICA PONDERADA SIN EXPANDIR LOS EVENTOS
def weighted_quantile(values, weights, p):
    idx = np.argsort(values)
    v = np.asarray(values, dtype=float)[idx]
    w = np.asarray(weights, dtype=np.int64)[idx]
    return v[min(np.searchsorted(np.cumsum(w), p*w.sum(), side="left"), len(v)-1)]

v = rd["rating"].to_numpy(dtype=float)
w = rd["frequency"].to_numpy(dtype=np.int64)
N = int(w.sum()); mean_event = np.average(v, weights=w)
var_event = np.dot(w, (v-mean_event)**2) / (N-1) if N>1 else np.nan
rating_event_stats = pd.DataFrame([{
    "n_eventos": N, "media": mean_event,
    "mediana": weighted_quantile(v,w,0.5), "moda": v[np.argmax(w)],
    "varianza_muestral": var_event, "desviacion_muestral": np.sqrt(var_event),
    "q1": weighted_quantile(v,w,0.25), "q3": weighted_quantile(v,w,0.75),
    "IQR": weighted_quantile(v,w,0.75)-weighted_quantile(v,w,0.25)
}])
print("Distribución de calificaciones por EVENTO")
display(rating_event_stats)
print("Distribución de medias por PELÍCULA, sólo títulos evaluados")
movie_rating_stats = pd.DataFrame([descriptive(movies.loc[movies["rating_count"]>0,"rating_mean"], "rating_mean", cv=False),
                                    descriptive(movies.loc[movies["rating_count"]>=100,"rating_mean"], "rating_mean (>=100 votos)", cv=False)])
display(movie_rating_stats.set_index("variable").T)
print("La varianza de las medias por película NO equivale a la varianza entre evaluaciones individuales.")

Distribución de calificaciones por EVENTO


,n_eventos,media,mediana,moda,varianza_muestral,desviacion_muestral,q1,q3,IQR
0,32000204,3.5404,3.5000,4.0000,1.1215,1.0590,3.0000,4.0000,1.0000


Distribución de medias por PELÍCULA, sólo títulos evaluados


variable,rating_mean,rating_mean (>=100 votos)
n,"84,432.0000","12,191.0000"
media,3.0051,3.2959
mediana,3.0714,3.3672
moda,3.0000,3.5000
modas_totales,1.0000,1.0000
varianza_muestral,0.6429,0.2301
desviacion_muestral,0.8018,0.4797
q1,2.5435,3.0137
q3,3.5000,3.6528
IQR,0.9565,0.6391


La varianza de las medias por película NO equivale a la varianza entre evaluaciones individuales.


## 5. Forma de distribución: asimetría y curtosis
La asimetría positiva indica cola hacia la derecha y la negativa, hacia la izquierda. Cuando la curtosisi reporta valores altos pueden reflejar colas pesadas o valores extremos, no necesariamente una forma normal.

In [32]:
# 7. FORMA ESTADÍSTICA
shape_rows = []
for c in count_vars + ["rating_mean"]:
    s = pd.to_numeric(movies[c], errors="coerce").dropna()
    if c == "rating_mean": s = s[movies.loc[s.index, "rating_count"] > 0]
    skew = s.skew() if len(s)>2 and s.nunique()>2 else np.nan
    kurt = s.kurt() if len(s)>3 and s.nunique()>3 else np.nan
    shape_rows.append({"variable":c,"n":len(s),"pct_ceros":100*(s==0).mean(),
                       "asimetria_fisher":skew,"orientacion": ("derecha" if skew>0 else "izquierda" if skew<0 else "simétrica aprox.") if pd.notna(skew) else "no evaluable",
                       "exceso_curtosis_fisher":kurt})
shape = pd.DataFrame(shape_rows)
display(shape)
# Momentos de la distribución de calificaciones individuales a partir de frecuencias.
mu = mean_event
m2 = np.average((v-mu)**2, weights=w)
m3 = np.average((v-mu)**3, weights=w)
m4 = np.average((v-mu)**4, weights=w)
print("Asimetría ponderada de calificaciones (momento poblacional):", m3/m2**1.5 if m2>0 else np.nan)
print("Exceso de curtosis ponderado de calificaciones (momento poblacional):", m4/m2**2-3 if m2>0 else np.nan)


,variable,n,pct_ceros,asimetria_fisher,orientacion,exceso_curtosis_fisher
0,rating_count,87585,3.5999,15.9020,derecha,360.2967
1,rating_users,87585,3.5999,15.9020,derecha,360.2967
2,tag_application_count,87585,41.4021,18.5215,derecha,592.1917
3,non_null_tag_count,87585,41.4021,18.5215,derecha,592.1949
4,distinct_tags,87585,41.4021,7.8836,derecha,82.2088
5,tag_users,87585,41.4021,24.6268,derecha,"1,469.3666"
6,rating_mean,84432,0.0000,-0.6101,izquierda,1.0645


Asimetría ponderada de calificaciones (momento poblacional): -0.7190238734384279
Exceso de curtosis ponderado de calificaciones (momento poblacional): 0.25944395897200456


## 6. Valores atípicos: detección, investigación y decisión

Se aplica la regla de Tukey `[Q1−1.5·RIQ, Q3+1.5·RIQ]`. Para datos muy concentrados en cero, RIQ puede ser cero y señalar muchas observaciones válidas. También se revisan los percentiles 95, 99 y 99.9. No se eliminan automáticamente películas populares ni títulos con pocas evaluaciones: la popularidad extrema puede ser genuina. Para puntuaciones individuales, 0.5 y 5 son límites válidos, no errores por sí mismos.


In [25]:
# 8. DETECCIÓN DE ATÍPICOS EN VARIABLES DE ACTIVIDAD
outlier_rows=[]
for c in count_vars:
    s=movies[c].dropna()
    q1,q3=s.quantile([.25,.75]); iqr=q3-q1; lo=max(0, q1-1.5*iqr); hi=q3+1.5*iqr
    outlier_rows.append({"variable":c,"q1":q1,"q3":q3,"IQR":iqr,
                         "limite_inferior":lo,"limite_superior":hi,"n_fuera":int(((s<lo)|(s>hi)).sum()),
                         "pct_fuera":100*((s<lo)|(s>hi)).mean(),
                         "p95":s.quantile(.95),"p99":s.quantile(.99),"p999":s.quantile(.999),
                         "maximo":s.max(),"IQR_cero":iqr==0})
outliers=pd.DataFrame(outlier_rows)
display(outliers)
print("Casos de máxima actividad para investigar; no se excluyen:")
display(movies.nlargest(12,"rating_count")[["movieId","title","rating_count","rating_mean","tag_application_count"]])
print("Casos con >=100 evaluaciones y dispersión interna más alta:")
display(movies.loc[movies["rating_count"]>=100].nlargest(10,"rating_std")[["movieId","title","rating_count","rating_mean","rating_std"]])

# Comprobar reglas de negocio antes de considerar un dato anómalo como error.
checks = {
    "conteos_negativos": int((movies[count_vars]<0).sum().sum()),
    "rating_mean_fuera_escala": int(((movies["rating_mean"].notna()) & ~movies["rating_mean"].between(0.5,5)).sum()),
    "media_sin_votos": int(((movies["rating_count"]==0)&movies["rating_mean"].notna()).sum()),
    "std_sin_al_menos_dos_votos": int(((movies["rating_count"]<2)&movies["rating_std"].notna()).sum()),
    "usuarios_mayores_que_calificaciones": int((movies["rating_users"]>movies["rating_count"]).sum()),
}
display(pd.Series(checks, name="incidencias"))

,variable,q1,q3,IQR,limite_inferior,limite_superior,n_fuera,pct_fuera,p95,p99,p999,maximo,IQR_cero
0,rating_count,2.0000,23.0000,21.0000,0,54.5000,15437,17.6252,"1,006.6000","8,877.8000","37,690.3440",102929,False
1,rating_users,2.0000,23.0000,21.0000,0,54.5000,15437,17.6252,"1,006.6000","8,877.8000","37,690.3440",102929,False
2,tag_application_count,0.0000,7.0000,7.0000,0,17.5000,12266,14.0047,85.0000,436.0000,"1,674.7280",6697,False
3,non_null_tag_count,0.0000,7.0000,7.0000,0,17.5000,12265,14.0035,85.0000,436.0000,"1,674.7280",6697,False
4,distinct_tags,0.0000,7.0000,7.0000,0,17.5000,10647,12.1562,50.0000,243.1600,519.0000,1111,False
5,tag_users,0.0000,2.0000,2.0000,0,5.0000,9008,10.2849,14.0000,71.0000,256.6640,2007,False


Casos de máxima actividad para investigar; no se excluyen:


,movieId,title,rating_count,rating_mean,tag_application_count
314,318,"Shawshank Redemption, The (1994)",102929,4.4046,4892
351,356,Forrest Gump (1994),100296,4.0527,3581
292,296,Pulp Fiction (1994),98409,4.1970,6697
2480,2571,"Matrix, The (1999)",93808,4.1564,5722
585,593,"Silence of the Lambs, The (1991)",90330,4.1484,3469
257,260,Star Wars: Episode IV - A New Hope (1977),85010,4.0998,6519
2867,2959,Fight Club (1999),77332,4.2288,5532
475,480,Jurassic Park (1993),75233,3.6986,1510
522,527,Schindler's List (1993),73849,4.2370,2059
4888,4993,"Lord of the Rings: The Fellowship of the Ring,...",73122,4.0921,2550


Casos con >=100 evaluaciones y dispersión interna más alta:


,movieId,title,rating_count,rating_mean,rating_std
1277,1311,Santa with Muscles (1996),163,2.5521,1.7075
14392,74754,"Room, The (2003)",1179,2.4822,1.6428
21478,110603,God's Not Dead (2014),241,2.4108,1.6277
12322,59295,Expelled: No Intelligence Allowed (2008),166,2.0964,1.5963
11344,50703,"Secret, The (2006)",204,2.3186,1.5738
1382,1421,Grateful Dead (1995),190,3.0421,1.5654
29196,133123,Barbie as The Princess & the Pauper (2004),142,3.2817,1.5419
1347,1383,Adrenalin: Fear the Rush (1996),149,2.2517,1.5264
52263,183589,Fifty Shades Freed (2018),394,2.4987,1.5206
13101,66659,Tyler Perry's Madea Goes to Jail (2009),142,2.5634,1.5134


conteos_negativos                      0
rating_mean_fuera_escala               0
media_sin_votos                        0
std_sin_al_menos_dos_votos             0
usuarios_mayores_que_calificaciones    0
Name: incidencias, dtype: int64

## 7. Calidad de datos y patrones de faltantes
Se evalúa número y porcentaje de nulos por variable. La tabla `column_quality` proviene de las fuentes **anteriores** a la consolidación y se muestra aparte: no se debe comparar directamente su denominador con el consolidado.


In [26]:
# 9. FALTANTES POR COLUMNA Y PATRONES CONJUNTOS
missing = pd.DataFrame({"variable":movies.columns,"faltantes":movies.isna().sum().values,
                        "pct_faltantes":movies.isna().mean().values*100,
                        "dtype":movies.dtypes.astype(str).values}).sort_values("faltantes",ascending=False)
display(missing)
print("Diagnóstico de faltantes estructurales en medidas de rating:")
display(pd.crosstab(movies["rating_count"].clip(upper=2).map({0:"0 evaluaciones",1:"1 evaluación",2:"2+ evaluaciones"}),
                    movies["rating_std"].isna(), rownames=["actividad"],colnames=["rating_std ausente"], margins=True))
print("Patrones combinados de ausencia: IDs externos y medias de rating")
cols = ["imdbId","tmdbId","rating_mean","rating_std"]
patterns = movies[cols].isna().rename(columns=lambda c:c+"_nulo")
display(patterns.value_counts(dropna=False).rename("peliculas").reset_index().head(20))
print("Comparación cobertura de TMDb según actividad:")
display(pd.crosstab(movies["has_ratings"], movies["tmdbId"].isna(), normalize="index"))
print("Informe de calidad preintegración:")
display(tables["column_quality"])
print("Resultados de validación y referencial:")
display(tables["post_validation"])
display(tables["referential_integrity"])

,variable,faltantes,pct_faltantes,dtype
8,rating_std,21760,24.8444,float64
10,rating_max,3153,3.5999,float64
7,rating_mean,3153,3.5999,float64
9,rating_min,3153,3.5999,float64
5,tmdbId,124,0.1416,float64
12,tag_application_count,0,0.0000,int64
18,has_imdb_id,0,0.0000,bool
17,has_tags,0,0.0000,bool
16,has_ratings,0,0.0000,bool
15,tag_users,0,0.0000,int64


Diagnóstico de faltantes estructurales en medidas de rating:


rating_std ausente,False,True,All
actividad,,,
0 evaluaciones,0,3153,3153
1 evaluación,0,18607,18607
2+ evaluaciones,65825,0,65825
All,65825,21760,87585


Patrones combinados de ausencia: IDs externos y medias de rating


,imdbId_nulo,tmdbId_nulo,rating_mean_nulo,rating_std_nulo,peliculas
0,False,False,False,False,65735
1,False,False,False,True,18575
2,False,False,True,True,3151
3,False,True,False,False,90
4,False,True,False,True,32
5,False,True,True,True,2


Comparación cobertura de TMDb según actividad:


... (2 outputs left out)

,dataset,column,dtype,non_null,nulls,null_pct,unique_non_null
0,movies,movieId,int32,87585,0,0.0000,87585
1,movies,title,string,87585,0,0.0000,87382
2,movies,genres,string,87585,0,0.0000,1798
3,ratings,userId,int32,32000204,0,0.0000,200948
4,ratings,movieId,int32,32000204,0,0.0000,84432
5,ratings,rating,float32,32000204,0,0.0000,10
6,ratings,timestamp,int64,32000204,0,0.0000,25976523
7,tags,userId,int32,2000072,0,0.0000,15848
8,tags,movieId,int32,2000072,0,0.0000,51323
9,tags,tag,string,2000055,17,0.0008,140979


Resultados de validación y referencial:


,test,passed
0,movieId único en movies_consolidated,True
1,Se preservó el total de películas,True
2,Combinación movieId-genre única,True
3,Géneros vinculados a películas existentes,True
4,Agregado de ratings reconcilia con ratings ori...,True
5,Agregado de tags reconcilia con tags original,True


,child_table,distinct_movie_ids,ids_found_in_movies,orphan_movie_ids
0,ratings,84432,84432,0
1,tags,51323,51323,0
2,links,87585,87585,0


## 8. Relaciones bivariadas, asociaciones y correlaciones

Se comparan métricas por película, nunca IDs. Las variables `rating_count` y `rating_users` pueden ser prácticamente equivalentes por diseño: una correlación elevada sería estructural, no un hallazgo independiente. Se omiten variables con varianza cero.

Las medias de calificación se correlacionan sólo entre películas con evaluaciones y se repite el análisis restringiendo a **≥100 evaluaciones**, porque una nota calculada con un voto tiene incertidumbre muy distinta de otra calculada con miles. Los resultados son descriptivos, no inferenciales ni causales.

In [27]:
# 10. CORRELACIONES DE VARIABLES DE ACTIVIDAD Y PUNTUACIÓN
variables = ["rating_count","tag_application_count","distinct_tags","tag_users","rating_mean","rating_std"]
full = movies.loc[movies["rating_count"]>0,variables].copy()
restricted = movies.loc[movies["rating_count"]>=100,variables].copy()
for name, subset in [("Películas con al menos 1 evaluación",full),("Películas con al menos 100 evaluaciones",restricted)]:
    print("\n",name,"n=",len(subset))
    print("Pearson:")
    display(subset.corr(method="pearson"))
    print("Spearman:")
    display(subset.corr(method="spearman"))

# N de observaciones completas para pares seleccionados (correlación sin n puede engañar).
pairs = [("rating_count","tag_application_count"),("rating_count","rating_mean"),
         ("rating_mean","rating_std"),("tag_application_count","distinct_tags")]
correlation_rows=[]
for filt, sub in [("n>=1",full),("n>=100",restricted)]:
    for a,b in pairs:
        z=sub[[a,b]].dropna()
        correlation_rows.append({"muestra":filt,"variable_x":a,"variable_y":b,"n_pares":len(z),
                                 "pearson":z[a].corr(z[b],method="pearson"),
                                 "spearman":z[a].corr(z[b],method="spearman")})
correlation_summary=pd.DataFrame(correlation_rows)
display(correlation_summary)
print("Correlaciones sensibles al tamaño muestral, a la selección de películas y a la distribución sesgada.")


 Películas con al menos 1 evaluación n= 84432
Pearson:


,rating_count,tag_application_count,distinct_tags,tag_users,rating_mean,rating_std
rating_count,1.0000,0.7689,0.5287,0.8381,0.0976,-0.0045
tag_application_count,0.7689,1.0000,0.7637,0.9332,0.1098,-0.0040
distinct_tags,0.5287,0.7637,1.0000,0.5987,0.1082,0.0045
tag_users,0.8381,0.9332,0.5987,1.0000,0.1198,-0.0056
rating_mean,0.0976,0.1098,0.1082,0.1198,1.0000,-0.1683
rating_std,-0.0045,-0.0040,0.0045,-0.0056,-0.1683,1.0000


Spearman:


,rating_count,tag_application_count,distinct_tags,tag_users,rating_mean,rating_std
rating_count,1.0000,0.6802,0.6737,0.7107,0.1549,0.1801
tag_application_count,0.6802,1.0000,0.9988,0.9570,0.1400,0.0579
distinct_tags,0.6737,0.9988,1.0000,0.9506,0.1371,0.0570
tag_users,0.7107,0.9570,0.9506,1.0000,0.1497,0.0580
rating_mean,0.1549,0.1400,0.1371,0.1497,1.0000,-0.2402
rating_std,0.1801,0.0579,0.0570,0.0580,-0.2402,1.0000



 Películas con al menos 100 evaluaciones n= 12191
Pearson:


,rating_count,tag_application_count,distinct_tags,tag_users,rating_mean,rating_std
rating_count,1.0000,0.7400,0.4676,0.8150,0.2135,-0.0875
tag_application_count,0.7400,1.0000,0.7269,0.9271,0.2500,-0.0965
distinct_tags,0.4676,0.7269,1.0000,0.5343,0.1751,-0.0564
tag_users,0.8150,0.9271,0.5343,1.0000,0.2768,-0.1168
rating_mean,0.2135,0.2500,0.1751,0.2768,1.0000,-0.5045
rating_std,-0.0875,-0.0965,-0.0564,-0.1168,-0.5045,1.0000


Spearman:


,rating_count,tag_application_count,distinct_tags,tag_users,rating_mean,rating_std
rating_count,1.0000,0.6233,0.5521,0.7520,0.1913,-0.1040
tag_application_count,0.6233,1.0000,0.9675,0.8534,0.3028,-0.1459
distinct_tags,0.5521,0.9675,1.0000,0.7518,0.2713,-0.1330
tag_users,0.7520,0.8534,0.7518,1.0000,0.3577,-0.1905
rating_mean,0.1913,0.3028,0.2713,0.3577,1.0000,-0.5135
rating_std,-0.1040,-0.1459,-0.1330,-0.1905,-0.5135,1.0000


,muestra,variable_x,variable_y,n_pares,pearson,spearman
0,n>=1,rating_count,tag_application_count,84432,0.7689,0.6802
1,n>=1,rating_count,rating_mean,84432,0.0976,0.1549
2,n>=1,rating_mean,rating_std,65825,-0.1683,-0.2402
3,n>=1,tag_application_count,distinct_tags,84432,0.7637,0.9988
4,n>=100,rating_count,tag_application_count,12191,0.7400,0.6233
5,n>=100,rating_count,rating_mean,12191,0.2135,0.1913
6,n>=100,rating_mean,rating_std,12191,-0.5045,-0.5135
7,n>=100,tag_application_count,distinct_tags,12191,0.7269,0.9675


Correlaciones sensibles al tamaño muestral, a la selección de películas y a la distribución sesgada.


### Asociación entre categorías y comportamiento por género
Se compara la cobertura de evaluaciones entre géneros mediante tasas. Para medias de puntuación, cada película se cuenta una vez **dentro de cada género** y se restringe a títulos con ≥100 votos; como las categorías se superponen, las filas no representan grupos independientes. Se muestra también mediana para robustez.

In [28]:
# 11. ASOCIACIONES CATEGÓRICAS Y COMPARACIÓN MULTILABEL
joint = pd.crosstab(movies["has_ratings"],movies["has_tags"],margins=True)
display(joint)
display(pd.crosstab(movies["has_ratings"],movies["has_tags"],normalize="index")*100)
per_genre = movie_genres.merge(
    movies[["movieId","rating_count","rating_mean","has_ratings","tag_application_count"]],
    on="movieId",how="left",validate="many_to_one")
genre_summary = per_genre.groupby("genre").agg(
    peliculas=("movieId","nunique"),
    pct_con_evaluaciones=("has_ratings",lambda x:100*x.mean()),
    mediana_evaluaciones=("rating_count","median"),
    media_evaluaciones=("rating_count","mean"),
    mediana_etiquetas=("tag_application_count","median"))
eligible = per_genre.loc[per_genre["rating_count"]>=100]
rating_genre = eligible.groupby("genre").agg(
    peliculas_con_100_votos=("movieId","nunique"),
    media_puntuacion_por_pelicula=("rating_mean","mean"),
    mediana_puntuacion_por_pelicula=("rating_mean","median"))
genre_summary=genre_summary.join(rating_genre)
display(genre_summary.sort_values("peliculas",ascending=False))
print("Porcentaje sobre catálogo por género NO suma 100%; pertenencias múltiples.")
print("No se calculan correlaciones entre nombres de géneros e IDs numéricos.")

has_tags,False,True,All
has_ratings,,,
False,0,3153,3153
True,36262,48170,84432
All,36262,51323,87585


has_tags,False,True
has_ratings,,
False,0.0000,100.0000
True,42.9482,57.0518


,peliculas,pct_con_evaluaciones,mediana_evaluaciones,media_evaluaciones,mediana_etiquetas,peliculas_con_100_votos,media_puntuacion_por_pelicula,mediana_puntuacion_por_pelicula
genre,,,,,,,,
Drama,34175,97.0066,6.0000,408.8741,2.0000,5772,3.4516,3.5110
Comedy,23124,97.0766,6.0000,484.6448,2.0000,4231,3.1927,3.2460
Thriller,11823,97.7332,7.0000,734.1169,3.0000,2498,3.2274,3.2517
Romance,10369,96.9042,6.0000,532.8011,2.0000,1919,3.3585,3.4161
Action,9668,96.1523,8.0000,999.7117,3.0000,2111,3.1522,3.1899
Documentary,9363,97.2231,3.0000,45.6427,1.0000,547,3.6476,3.7202
Horror,8654,97.8507,6.0000,287.9957,3.0000,1357,2.9776,3.0322
(no genres listed),7080,92.5847,2.0000,7.8387,0.0000,47,3.4514,3.5340
Crime,6976,96.1009,7.0000,770.2195,3.0000,1469,3.3541,3.4105


Porcentaje sobre catálogo por género NO suma 100%; pertenencias múltiples.
No se calculan correlaciones entre nombres de géneros e IDs numéricos.


## 9. Conclusiones
Se redacta un resumen a partir de lo calculado.

In [29]:
# 12. SÍNTESIS AUTOMÁTICA DE RESULTADOS
n_movies=len(movies)
n_rated=int(movies["has_ratings"].sum())
n_tagged=int(movies["has_tags"].sum())
median_v=float(movies["rating_count"].median())
mean_v=float(movies["rating_count"].mean())
mode_rating=float(rd.loc[rd["frequency"].idxmax(),"rating"])
rank_corr=correlation_summary.query('muestra == "n>=100" and variable_x == "rating_count" and variable_y == "rating_mean"')["spearman"].iloc[0]
most_genre=genre_summary["peliculas"].idxmax()
print("¿QUÉ REPRESENTAN LOS REGISTROS MÁS IMPORTANTES?")
print(f"• Catálogo: {n_movies:,} películas, {n_rated:,} con evaluaciones y {n_tagged:,} con etiquetas.")
print(f"• Distribución: {N:,} eventos de rating; puntuación modal {mode_rating:g} y media por evento {mean_event:.3f}.")
print("• Las películas con más eventos documentan concentración de actividad; sin eventos, no se puede estimar opinión.")
print("\n¿QUÉ VARIABLES SE TIENEN?")
print(f"• {len(movies.columns)} columnas de catálogo: identificadores y etiquetas nominales, flags dicotómicos, conteos de razón y resúmenes de rating.")
print("• Géneros: variable nominal multilabel; rating original: ordinal discreta (intervalo aproximado para promedios).")
print("\n¿QUÉ PATRONES SE DETECTARON?")
print(f"• Votos por película: media {mean_v:.2f} y mediana {median_v:.2f}; la diferencia indica concentración de actividad." )
print(f"• Género con mayor presencia: {most_genre} ({int(genre_summary.loc[most_genre,'peliculas']):,} películas).")
print(f"• Asociación monotónica votos–media con >=100 votos (Spearman): {rank_corr:.3f}.")
print(f"• Se localizaron nulos en {int((missing['faltantes']>0).sum())} columnas, con diagnóstico de ausencia estructural y cobertura.")
print("• Los outliers se conservan, sujetos a revisión de reglas de negocio; no hay recodificación automática.")
print("\nLIMITACIONES")
print("• Sin datos originales de usuario y fecha: no se evalúan efectos temporales, cohortes o dependencia intrausuario.")
print("• Un género no es exclusivo y una correlación no demuestra causalidad.")

¿QUÉ REPRESENTAN LOS REGISTROS MÁS IMPORTANTES?
• Catálogo: 87,585 películas, 84,432 con evaluaciones y 51,323 con etiquetas.
• Distribución: 32,000,204 eventos de rating; puntuación modal 4 y media por evento 3.540.
• Las películas con más eventos documentan concentración de actividad; sin eventos, no se puede estimar opinión.

¿QUÉ VARIABLES SE TIENEN?
• 20 columnas de catálogo: identificadores y etiquetas nominales, flags dicotómicos, conteos de razón y resúmenes de rating.
• Géneros: variable nominal multilabel; rating original: ordinal discreta (intervalo aproximado para promedios).

¿QUÉ PATRONES SE DETECTARON?
• Votos por película: media 365.36 y mediana 5.00; la diferencia indica concentración de actividad.
• Género con mayor presencia: Drama (34,175 películas).
• Asociación monotónica votos–media con >=100 votos (Spearman): 0.191.
• Se localizaron nulos en 5 columnas, con diagnóstico de ausencia estructural y cobertura.
• Los outliers se conservan, sujetos a revisión de reglas